# H=3 · Kho biến rộng và hai test rolling

Notebook chạy độc lập từ Parquet MBD-mini. Các cell 01–07 phát triển và khóa mô hình. Cell 08 chấm test theo cấu hình khóa. Cell 09 tổng hợp để lưu cùng notebook.

**Tiến độ.** Từng nguồn/tháng và lượt fit đều in thời gian. Tiến độ fold, lượt bootstrap và số kết quả đã cache hiển thị trực tiếp.

## 01 · Cài thư viện và thiết lập phiên chạy

Mở **Runtime → Change runtime type → GPU** để chạy đủ dữ liệu. Đặt `SMOKE=True` để kiểm tra nhanh trên CPU hoặc GPU. Đặt `LABELS_READY_NEXT_MONTH=True` chỉ sau khi xác nhận nhãn tháng trước đã có tại cutoff. Mặc định Val tháng 5 được bỏ qua vì nhãn train chưa đủ.

In [ ]:
%pip -q install "duckdb>=1.4,<3" "catboost>=1.2.8,<2" "xgboost>=3,<4" "lightgbm>=4.5,<5" huggingface_hub
import os, sys, gc, json, time, threading, hashlib, subprocess
from contextlib import contextmanager
from pathlib import Path
from itertools import combinations
import duckdb as db
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

SMOKE = False                  # True: ít khách, số cây thấp, kiểm tra toàn luồng
LABELS_READY_NEXT_MONTH = False # Chỉ đổi True nếu xác nhận nhãn có trước ngày 01
USE_DRIVE = True              # Lưu checkpoint để có thể tiếp tục sau khi mất phiên
RUN_LOCKED_TEST = False       # Chạy Val trước. Sau khi xem cấu hình khóa, bật ở Cell 08
DATA_DIR = Path('/content/MBD-mini')
WORK = Path('/content/h3-wide')
if USE_DRIVE and 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/h3-wide')
months = list(pd.period_range('2022-02', '2023-01', freq='M').astype(str))
val_months = months[3:6]
test_months = [months[8], months[9]]
fit_months = {m: months[:months.index(m)-2-int(not LABELS_READY_NEXT_MONTH)]
              for m in val_months + test_months if months.index(m)-2-int(not LABELS_READY_NEXT_MONTH) > 0}
if not LABELS_READY_NEXT_MONTH:
    print('Nhãn trễ mặc định (lag=1): Val tháng 5 bỏ qua; các snapshot train của Val (<= 2022-03) chưa có lịch sử prior_* quan sát được; train Test 10 có 5 snapshot.', flush=True)
gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip() if __import__('shutil').which('nvidia-smi') else ''
if not gpu and not SMOKE:
    raise RuntimeError('Chọn Runtime → Change runtime type → GPU rồi chạy lại.')
print(gpu or 'Smoke chạy CPU', flush=True)

def say(message):
    print(time.strftime('%H:%M:%S'), message, flush=True)

@contextmanager
def stage(name, interval=45):
    start = time.monotonic()
    done = threading.Event()
    def heartbeat():
        while not done.wait(interval):
            say(f'{name} · đang chạy {time.monotonic()-start:.0f}s')
    say(f'Bắt đầu {name}')
    threading.Thread(target=heartbeat, daemon=True).start()
    try:
        yield
    except Exception:
        say(f'Lỗi {name} sau {time.monotonic()-start:.1f}s')
        raise
    else:
        say(f'Xong {name} · {time.monotonic()-start:.1f}s')
    finally:
        done.set()

def key(*values):
    return hashlib.sha256(json.dumps(values, sort_keys=True, default=str).encode()).hexdigest()[:20]

def save(frame, path):
    tmp = path.with_suffix('.tmp.parquet')
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)


## 02 · Lấy dữ liệu và kiểm tra lịch

Kiểm tra schema, tải dữ liệu từ kho MBD-mini nếu chưa có, tạo thư mục checkpoint trên Drive và in lịch train/Val. Có thể dùng thư mục Parquet đã chuẩn bị bằng cách chỉnh `DATA_DIR`.

In [ ]:
if not all((DATA_DIR/name).exists() for name in ('client_split', 'targets', 'detail')):
    import shutil, tarfile
    from huggingface_hub import hf_hub_download
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name in ('client_split', 'targets', 'detail'):
        if (DATA_DIR/name).exists():
            continue
        with stage(f'Tải và giải nén {name}.tar.gz'):
            arc = Path(hf_hub_download(repo_id='ai-lab/MBD-mini', filename=f'{name}.tar.gz', repo_type='dataset'))
            with tarfile.open(arc, 'r:gz') as tf:
                first = ''
                while (m := tf.next()) is not None:
                    cand = m.name.lstrip('./')
                    if cand:
                        first = cand
                        break
            dest = DATA_DIR if (first == name or first.startswith(name + '/')) else (DATA_DIR/name)
            dest.mkdir(parents=True, exist_ok=True)
            if shutil.which('tar'):
                subprocess.run(['tar', '-xzf', str(arc), '-C', str(dest)], check=True)
            else:
                with tarfile.open(arc, 'r:gz') as tf:
                    tf.extractall(dest, filter='data') if sys.version_info >= (3, 12) else tf.extractall(dest)
def files(pattern):
    paths = sorted(DATA_DIR.glob(pattern))
    if not paths:
        raise FileNotFoundError(f'Thiếu dữ liệu {DATA_DIR / pattern}')
    return paths

inputs = {name: files(pattern) for name, pattern in {
    'trx':'detail/trx/*/*.parquet', 'geo':'detail/geo/*/*.parquet',
    'dialog':'detail/dialog/*/*.parquet', 'targets':'targets/*/*.parquet',
    'folds':'client_split/*/*.parquet'}.items()}
# Fingerprint ổn định khi tải lại cùng Parquet trên runtime mới.
from types import CodeType
from importlib.metadata import version

def code_fingerprint(*functions):
    def constant(value):
        if isinstance(value,CodeType):
            return signature(value)
        if isinstance(value,frozenset):
            return sorted(map(repr,value))
        return repr(value)

    def signature(code):
        return (code.co_code.hex(),code.co_names,[constant(v) for v in code.co_consts])

    return key(*(signature(fn.__code__) for fn in functions))

def fingerprint(p):
    size=p.stat().st_size
    with p.open('rb') as handle:
        digest = hashlib.file_digest(handle,'sha256').hexdigest()
    return str(p.relative_to(DATA_DIR)),size,digest
with stage('Nhận dạng dữ liệu và checkpoint'):
    BASE_RUN = WORK/key('wide-v2', SMOKE, LABELS_READY_NEXT_MONTH, bool(gpu),
                        fit_months, sys.version_info[:2], [version(pkg) for pkg in ('duckdb','pandas','numpy')],
                        [fingerprint(p) for paths in inputs.values() for p in paths])
BASE_RUN.mkdir(parents=True, exist_ok=True)
RUN = BASE_RUN
db.execute("SET memory_limit='3GB'; SET threads=2; SET preserve_insertion_order=false")
db.execute("SET temp_directory='/content/h3-duckdb'" if Path('/content').exists() else "SET temp_directory='h3-duckdb-scratch'")
def scan(name):
    names = ','.join("'" + str(p).replace("'", "''") + "'" for p in inputs[name])
    return f'read_parquet([{names}], hive_partitioning=true)'
sample = 'QUALIFY row_number() OVER (PARTITION BY fold ORDER BY hash(client_id)) <= 600' if SMOKE else ''
db.execute(f'CREATE OR REPLACE TEMP TABLE cohort AS SELECT DISTINCT client_id, CAST(fold AS INTEGER) AS fold FROM {scan("folds")} {sample}')
cohort = db.table('cohort').df()
assert not cohort.client_id.duplicated().any() and set(cohort.fold)==set(range(5))
active_val = [m for m in val_months if m in fit_months]
if len(active_val) < 2:
    raise RuntimeError('Cần ít nhất hai mốc Val có nhãn train đã đóng.')
for m in active_val + test_months[:1]:
    assert all(pd.Period(m).ordinal-pd.Period(t).ordinal >= 3+int(not LABELS_READY_NEXT_MONTH) for t in fit_months[m])
assert set(months[3:8]).isdisjoint(months[8:])
display(pd.DataFrame([(m, ', '.join(fit_months.get(m, [])) or 'Bỏ qua', ', '.join(months[months.index(m):months.index(m)+3]))
                      for m in val_months+test_months[:1]], columns=['Mốc','Snapshot train','Nhãn H=3']))
say(f'{len(cohort):,} khách · cache {RUN}')


## 03 · Dựng panel theo snapshot và ghi checkpoint sau mỗi nguồn/tháng

Tạo đặc trưng trước cutoff Moscow theo từng nguồn và tháng. Ô dài sẽ in thông báo mỗi 45 giây, sau đó lưu Parquet để chạy lại nhanh hơn.

In [ ]:
windows = (7, 14, 30, 60, 90, 180, None)
products = [f'target_{j}' for j in range(1,5)]
categories = {
    'trx':['src_type11','src_type12','src_type21','src_type22','src_type31','src_type32',
           'dst_type11','dst_type12','event_type','event_subtype','currency'],
    'geo':['geohash_4','geohash_5','geohash_6'],
    'dialog':[]
}
def target_rows(allowed):
    values = ','.join(repr(m) for m in allowed)
    cols = ','.join(f'COALESCE(t.{p},0)::TINYINT AS {p}' for p in products)
    t = db.sql(f'SELECT t.client_id, substr(t.mon,1,7) AS mon, {cols} FROM {scan("targets")} t JOIN cohort c USING(client_id) WHERE substr(t.mon,1,7) IN ({values})').df()
    if t.duplicated(['client_id','mon']).any():
        raise ValueError('Target bị trùng client_id × tháng')
    return t

# Kết quả tháng 10/11 chưa được nạp vào bộ chọn; hàm này chỉ lấy lịch sử đã có trước cutoff.
def prior_panel(m, lag):
    i = months.index(m)
    previous = months[:max(0,i-lag)]
    idx = pd.Index(cohort.client_id)
    hist = np.full((len(idx),len(previous),4),np.nan,np.float32)
    if previous:
        rows = target_rows(previous)
        hist[idx.get_indexer(rows.client_id), pd.Index(previous).get_indexer(rows.mon)] = rows[products]
    observed = np.isfinite(hist[:,:,0])
    base = {'client_id':idx, 'raw_prior_observed':observed.sum(1)}
    advanced = {}
    for j in range(4):
        x = hist[:,:,j]
        buy = x == 1
        base[f'raw_prior_p{j+1}_ever'] = np.where(observed.any(1),buy.any(1),np.nan)
        positions = np.arange(1,len(previous)+1)
        first = np.min(np.where(buy, positions, np.inf), axis=1, initial=np.inf)
        last = np.max(np.where(buy, positions, -np.inf), axis=1, initial=-np.inf)
        base[f'raw_prior_p{j+1}_first_recency'] = np.where(buy.any(1), len(previous)+1+lag-first, np.nan)
        advanced[f'prior_p{j+1}_purchase_span'] = np.where(buy.sum(1)>1, last-first, np.nan)
        for offset in (1,2,3,6):
            base[f'raw_prior_p{j+1}_lag{offset}'] = x[:,-offset] if len(previous)>=offset else np.nan
        recency = np.where(buy, np.arange(len(previous),0,-1)+lag, np.inf)
        base[f'raw_prior_p{j+1}_recency'] = np.min(recency,axis=1,initial=np.inf)
        for w in (1,2,3,6,99):
            seen = observed[:,-w:].sum(1)
            count = buy[:,-w:].sum(1)
            base[f'raw_prior_p{j+1}_{w}m'] = np.where(seen>0,count,np.nan)
            advanced[f'prior_rate_p{j+1}_{w}m'] = np.divide(count,seen,out=np.full(len(idx),np.nan),where=seen>0)
    for a,b in combinations(range(4),2):
        positions = np.arange(len(previous))
        first_a = np.min(np.where(hist[:,:,a]==1, positions, np.inf), axis=1, initial=np.inf)
        first_b = np.min(np.where(hist[:,:,b]==1, positions, np.inf), axis=1, initial=np.inf)
        both = np.isfinite(first_a) & np.isfinite(first_b)
        advanced[f'prior_first_p{a+1}_before_p{b+1}'] = np.where(both, first_a < first_b, np.nan)
        advanced[f'prior_co_buy_{a+1}_{b+1}'] = np.where(observed.any(1),((hist[:,:,a]==1)&(hist[:,:,b]==1)).sum(1),np.nan)
    advanced['prior_product_diversity'] = np.where(observed.any(1), (hist==1).any(axis=1).sum(axis=1), np.nan)
    return pd.DataFrame({**base,**advanced}).replace(np.inf,np.nan)

def source_panel(source,m,mode='main'):
    path = FEATURE_RUN/f'{source}_{m}_{mode}.parquet'
    if path.exists():
        say(f'Cache {path.name}')
        return pd.read_parquet(path)
    cutoff = f"TIMESTAMP '{m}-01'"
    shift = '0' if mode=='utc' else '3'
    delay = int(mode.removeprefix('lag')) if mode.startswith('lag') else 0
    end = f"({cutoff} - INTERVAL {delay} DAY)"
    with stage(f'{source} {m} {mode}'):
        scan_source = scan(source)
        db.execute(f'''CREATE OR REPLACE TEMP VIEW events AS SELECT r.*, r.event_time + INTERVAL {shift} HOUR AS ts
                      FROM {scan_source} r JOIN cohort c USING(client_id)
                      WHERE r.event_time + INTERVAL {shift} HOUR < {end}''')
        expr = []
        for w in windows:
            name = f'raw_{source}_{w or "all"}'
            cond = f'ts >= {cutoff} - INTERVAL {w} DAY' if w else 'true'
            for suffix,sql in {
                'count':f'count(*) FILTER (WHERE {cond})',
                'days':f'count(DISTINCT CAST(ts AS DATE)) FILTER (WHERE {cond})',
                'recency':f'epoch({cutoff}-max(ts) FILTER (WHERE {cond}))/86400',
                'tenure':f'epoch({cutoff}-min(ts) FILTER (WHERE {cond}))/86400',
            }.items():
                expr.append(f'({sql})::FLOAT AS {name}_{suffix}')
            for col in categories[source]:
                expr.append(f'count(DISTINCT {col}) FILTER (WHERE {cond})::FLOAT AS {name}_{col}_unique')
            if source=='trx':
                for suffix, agg in [('sum','sum(amount)'),('avg','avg(amount)'),('max','max(amount)'),('std','stddev_pop(amount)')]:
                    expr.append(f'{agg} FILTER (WHERE {cond} AND currency=11 AND isfinite(amount))::FLOAT AS {name}_currency11_{suffix}')
        part = db.sql(f'SELECT client_id,{",".join(expr)} FROM events GROUP BY client_id').df()
        if mode=='main':
            if categories[source]:
                cat_cols = ','.join(categories[source])
                db.execute(f'''CREATE OR REPLACE TEMP TABLE events_90 AS
                              SELECT client_id, ts, {cat_cols} FROM events
                              WHERE ts >= {cutoff} - INTERVAL 90 DAY''')
            if source in {'trx','geo','dialog'}:
                code = {'trx':'event_type','geo':'geohash_5','dialog':'CAST(NULL AS INTEGER)'}[source]
                src_90 = 'events_90' if categories[source] else f'(SELECT client_id, ts FROM events WHERE ts >= {cutoff}-INTERVAL 90 DAY)'
                order_expr = 'ORDER BY ts' if source=='dialog' else f'ORDER BY ts, {code}'
                behavior = db.sql(f'''WITH ordered AS (
                    SELECT client_id, ts, {code} AS code,
                           lag(ts) OVER (PARTITION BY client_id {order_expr}) AS prev_ts,
                           lag({code}) OVER (PARTITION BY client_id {order_expr}) AS prev_code
                    FROM {src_90}
                ) SELECT client_id,
                         avg(epoch(ts-prev_ts)/86400)::FLOAT AS {source}_avg_gap,
                         count(*) FILTER (WHERE ts-prev_ts <= INTERVAL 1 DAY)::FLOAT AS {source}_bursts,
                         avg((code<>prev_code)::INTEGER)::FLOAT AS {source}_transition_rate,
                         avg((extract(isodow FROM ts)>=6)::INTEGER)::FLOAT AS {source}_weekend_share,
                         avg((extract(hour FROM ts) BETWEEN 8 AND 18)::INTEGER)::FLOAT AS {source}_daytime_share
                  FROM ordered GROUP BY client_id''').df()
                part = part.merge(behavior,on='client_id',how='left',validate='one_to_one')
            if categories[source]:
                pairs = [(a,b) for a in categories[source] if a.startswith('src_')
                         for b in categories[source] if b.startswith('dst_')]
                codes = {col: col for col in categories[source]}
                codes.update({f'{a}_{b}': f"CASE WHEN {a} IS NOT NULL AND {b} IS NOT NULL THEN concat_ws(':',{a},{b}) END"
                              for a,b in pairs})
                for col,code in codes.items():
                    for w in (30,90):
                        p = f'{source}_{col}_{w}'
                        w_cond = f'ts >= {cutoff}-INTERVAL {w} DAY AND ' if w < 90 else ''
                        q = f"""WITH c AS (SELECT client_id, {code} AS code, count(*) AS n
                                           FROM events_90 WHERE {w_cond}({code}) IS NOT NULL GROUP BY 1,2),
                                       s AS (SELECT *, n::DOUBLE/sum(n) OVER(PARTITION BY client_id) AS p FROM c)
                                SELECT client_id, (-sum(p*ln(p)))::FLOAT AS {p}_entropy,
                                       max(p)::FLOAT AS {p}_top_share FROM s GROUP BY 1"""
                        part = part.merge(db.sql(q).df(),on='client_id',how='left',validate='one_to_one')
                db.execute('DROP TABLE IF EXISTS events_90')
            if source=='dialog':
                r = db.sql(f'''SELECT client_id, ts, embedding FROM events
                             WHERE embedding IS NOT NULL AND ts >= {cutoff}-INTERVAL 180 DAY''').df()
                if len(r):
                    valid = r.embedding.map(lambda v: np.size(v)==768 and np.isfinite(v).all())
                    r = r.loc[valid].reset_index(drop=True)
                if len(r):
                    a = np.vstack(r.embedding).astype('float32')
                    # Một phép chiếu ngẫu nhiên cố định, không fit trên dữ liệu hay target.
                    projection = np.random.default_rng(42).normal(0,1/np.sqrt(32),(a.shape[1],32)).astype('float32')
                    z = a @ projection
                    age = (pd.Timestamp(m+'-01')-r.ts.dt.tz_localize(None)).dt.total_seconds().to_numpy()/86400
                    weights = np.exp(-age/30).astype('float32')
                    e = pd.DataFrame(z*weights[:,None], columns=[f'dlg_emb_{i:02}' for i in range(32)])
                    e['client_id'],e['weight'] = r.client_id,weights
                    sums = e.groupby('client_id',sort=False).sum()
                    emb = sums.drop(columns='weight').div(sums.weight,axis=0).reset_index()
                    part = part.merge(emb,on='client_id',how='left',validate='one_to_one')
                    plain = pd.DataFrame(z,columns=e.columns[:32]).assign(client_id=r.client_id)
                    vectors = plain.groupby('client_id',sort=False)
                    summary = [vectors.mean().add_suffix('_mean'),
                               plain.assign(ts=r.ts).sort_values('ts').drop_duplicates('client_id',keep='last')
                                    .set_index('client_id').drop(columns='ts').add_suffix('_last')]
                    recent = plain.loc[age<=30].groupby('client_id').mean()
                    history = plain.loc[age>30].groupby('client_id').mean()
                    recent,history = recent.align(history,join='inner',axis=0)
                    norm = np.sqrt(recent.pow(2).sum(axis=1)*history.pow(2).sum(axis=1))
                    summary.append(recent.mul(history).sum(axis=1).div(norm.where(norm>0)).rename('dlg_emb_cosine'))
                    summary.append(vectors.var().mean(axis=1).rename('dlg_emb_dispersion'))
                    part = part.merge(pd.concat(summary,axis=1).reset_index(),on='client_id',how='left',validate='one_to_one')
        save(part,path)
        return part

def panel(m, variant='main'):
    path = FEATURE_RUN/f'panel_{m}_{variant}.parquet'
    if path.exists():
        return pd.read_parquet(path)
    lag = int(not LABELS_READY_NEXT_MONTH) + int(variant=='prior_delay')
    use_variant = variant in {'utc','lag7','lag14'}
    # Bản kiểm chứng thay đúng khối biến thô tương ứng, giữ nhóm khác bất biến.
    frame = cohort.merge(prior_panel(m,lag),on='client_id',how='left',validate='one_to_one')
    for source in categories:
        mode = variant if use_variant and ((variant=='utc' and source=='trx') or (variant.startswith('lag') and source=='dialog')) else 'main'
        frame = frame.merge(source_panel(source,m,mode),on='client_id',how='left',validate='one_to_one')
    raw_counts = [c for c in frame if c.startswith('raw_') and (c.endswith(('_count','_days','_unique')))]
    frame = frame.fillna({c:0 for c in raw_counts}).copy()
    for source in categories:
        a,b = frame[f'raw_{source}_30_count'],frame[f'raw_{source}_90_count']
        frame[f'x_{source}_30_share_90'] = a / b.where(b>0)
    frame['x_no_trx_has_dialog'] = ((frame.raw_trx_90_count==0)&(frame.raw_dialog_30_count>0)).astype('float32')
    frame['x_geo_dlg'] = np.log1p(frame.raw_geo_30_count)*np.log1p(frame.raw_dialog_30_count)
    frame['x_geo_shift_prior'] = (frame.raw_geo_30_count-frame.raw_geo_90_count/3) * frame.filter(regex=r'^raw_prior_p[1-4]_3m$').sum(axis=1,min_count=1)
    frame['x_trx_return'] = ((frame.raw_trx_30_count>0)&(frame.raw_trx_30_count==frame.raw_trx_90_count)).astype('float32')
    frame['x_cross_source_change'] = np.prod([frame[f'x_{s}_30_share_90']-1/3 for s in categories],axis=0)
    if variant=='cap_tenure':
        tenure = [c for c in frame if c.endswith('_tenure')]
        frame[tenure] = frame[tenure].clip(upper=365)
    frame['month'] = m
    names = frame.columns.difference(['client_id','month','fold'])
    frame[names] = frame[names].astype('float32')
    save(frame,path)
    return frame

def labels(anchors):
    universe = sorted({v for m in anchors for v in months[months.index(m):months.index(m)+3]})
    y = target_rows(universe)
    y['y_any'] = y[products].max(axis=1)
    wide = y.pivot(index='client_id',columns='mon',values='y_any').reindex(columns=universe)
    parts = {}
    for m in anchors:
        values = wide[months[months.index(m):months.index(m)+3]]
        high = values.max(axis=1)
        parts[m] = high.where(high.eq(1)|values.notna().all(axis=1)).rename('y')
    return parts

FEATURE_RUN = BASE_RUN/key(windows,categories,products,code_fingerprint(target_rows,prior_panel,source_panel,panel,labels))
FEATURE_RUN.mkdir(parents=True,exist_ok=True)
dev = sorted(set(active_val).union(*(fit_months[m] for m in active_val)))
dev_labels = labels(dev)
for m in tqdm(dev, desc='Panel phát triển', unit='tháng'):
    panel(m)


## 04 · Hàm fit, metric và checkpoint theo fold

Khai báo common loop, khóa fold theo khách, chọn biến bên trong train, huấn luyện số cây cố định. Mỗi fold có thanh tiến độ, log thời gian và checkpoint dự đoán.

In [ ]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss, brier_score_loss

threads = min(4, os.cpu_count() or 1)
tree_default = 12 if SMOKE else 400
BASE = {'family':'cat','groups':[],'k':None,'trees':tree_default,'depth':6,'prior':False}
def cfg(**changes):
    return {**BASE, **changes}

def feature_group(name):
    if name.startswith('raw_'): return 'raw'
    if name.startswith('prior_'): return 'purchase'
    if name.startswith('dlg_emb_'): return 'embedding'
    if name.startswith('x_'): return 'interactions'
    return name.split('_',1)[0]

def learner(conf, seed):
    family = conf['family']
    trees, depth = conf['trees'], conf['depth']
    if family=='cat':
        return CatBoostClassifier(iterations=trees, depth=depth, learning_rate=.05, l2_leaf_reg=20,
                                  random_seed=seed, thread_count=threads, allow_writing_files=False,
                                  task_type='GPU' if gpu else 'CPU', verbose=max(1,trees//4))
    if family=='xgb':
        return XGBClassifier(n_estimators=trees,max_depth=depth,learning_rate=.05,reg_lambda=20,
                             tree_method='hist',device='cuda' if gpu else 'cpu',random_state=seed,n_jobs=threads)
    if family=='lgb':
        return LGBMClassifier(n_estimators=trees,max_depth=depth,num_leaves=min(31,2**depth-1),
                              learning_rate=.05,max_bin=127,colsample_bytree=.65,
                              reg_lambda=20,random_state=seed,n_jobs=threads,verbosity=-1)
    return make_pipeline(SimpleImputer(strategy='constant',fill_value=0,add_indicator=True),
                         StandardScaler(),LogisticRegression(max_iter=150,random_state=seed))

def cols_train(tr, conf, variant):
    candidates = sorted(c for c in tr if feature_group(c) in ['raw',*conf['groups']]
                        and not (variant=='no_prior' and c.startswith('raw_prior'))
                        and not (variant=='no_dialog' and c.startswith('raw_dialog'))
                        and not (variant=='no_geo' and c.startswith('raw_geo')))
    x = tr[candidates].replace([np.inf,-np.inf],np.nan)
    cols = [c for c in x if x[c].count() >= .0005*len(x)
            and x[c].nunique(dropna=False)>1]
    # Hash cột để lọc trùng chính xác mà không tạo ma trận tương quan kích thước p².
    seen = set()
    unique = []
    for c in cols:
        stamp = hashlib.blake2b(x[c].to_numpy().tobytes(),digest_size=16).digest()
        if stamp not in seen:
            unique.append(c)
            seen.add(stamp)
    return unique

def metrics(frame):
    y,s = frame.y.to_numpy(),frame.score.to_numpy()
    if len(np.unique(y)) < 2: raise ValueError('Mốc đánh giá phải có cả hai lớp')
    s = np.clip(s,1e-7,1-1e-7)
    n = int(np.ceil(.01*len(y)))
    ties = pd.util.hash_pandas_object(frame.client_id,index=False).to_numpy()
    top = np.lexsort((ties,-s))[:n]
    precision = y[top].mean()
    return dict(base_rate=y.mean(),PR_AUC=average_precision_score(y,s),
                precision_1=precision,lift_1=precision/y.mean(),ROC_AUC=roc_auc_score(y,s),
                LogLoss=log_loss(y,s),Brier=brier_score_loss(y,s))

def attach(frame, ys):
    return frame.join(ys,on='client_id').dropna(subset='y')

def rule(frame):
    f = frame.filter(regex=r'^raw_prior_p[1-4]_3m$').fillna(0).sum(axis=1)
    f += frame.raw_trx_30_count.fillna(0).rank(pct=True)+frame.raw_dialog_30_count.fillna(0).rank(pct=True)
    return f.rank(pct=True).to_numpy()

fit_events = []
records = []
decisions = []
def decision(stage_name, candidate, accepted, reason):
    decisions.append(dict(stage=stage_name,candidate=str(candidate),accepted=accepted,reason=reason))
    pd.DataFrame(decisions).to_csv(RUN/'selection_decisions.csv',index=False)

def evaluate_permutation(group, eval_months, ys):
    conf = cfg(groups=[group])
    original = predict_config(conf,eval_months,ys)
    permuted = predict_config(conf,eval_months,ys,variant=f'permute_{group}')
    delta = pd.DataFrame([
        {'month':m, **{name: value-metrics(permuted.loc[permuted.month.eq(m)])[name]
                      for name,value in metrics(part).items() if name in {'PR_AUC','lift_1'}}}
        for m,part in original.groupby('month')
    ])
    display(delta.assign(group=group))
    return float(delta.PR_AUC.mean())

def record_fit(anchor,fold,seed,family,started):
    elapsed = round(time.monotonic()-started,2)
    say(f'Fit {family} {anchor} f{fold} seed={seed} · {elapsed:.2f}s')
    fit_events.append(dict(anchor=anchor,fold=fold,seed=seed,family=family,fit_seconds=elapsed))
    pd.DataFrame(fit_events).to_csv(RUN/'fit_log.csv',index=False)

def predict_config(conf, eval_months, ys, seed=42, variant='main', locked_test=False):
    parts, n_feats = [], []
    permuted_group = variant.removeprefix('permute_') if variant.startswith('permute_') else None
    panel_variant = 'main' if permuted_group else variant
    batches = [(test_months[0],eval_months)] if locked_test else [(m,[m]) for m in eval_months]
    bar = tqdm(total=len(batches)*5,desc=f'{conf["family"]} K={conf["k"]} seed={seed} {variant}',unit='fold')
    try:
        for anchor, scoring in batches:
            tm = fit_months[anchor]
            if variant=='step3':
                tm = tm[::-3][::-1]
            train_all = eval_all = None
            for fold in range(5):
                token = key(conf,seed,variant,anchor,tm,scoring,fold)
                dest = RUN/f'pred_{token}.parquet'
                feat_path = RUN/f'features_{token}.json'
                bar.set_postfix_str(f'{anchor} f{fold}'+(' cache' if dest.exists() else ''))
                if dest.exists():
                    parts.append(pd.read_parquet(dest))
                    if feat_path.exists():
                        n_feats.append(len(json.loads(feat_path.read_text(encoding='utf8'))))
                else:
                    with stage(f'Fit/chấm {anchor} f{fold} · {conf["family"]}'):
                        if train_all is None:
                            train_all = pd.concat([attach(panel(m,panel_variant),ys[m]) for m in tm],ignore_index=True)
                            eval_all = [attach(panel(m,panel_variant),ys[m]) for m in scoring]
                        tr = train_all.loc[train_all.fold.ne(fold)]
                        evs = [ev.loc[ev.fold.eq(fold)] for ev in eval_all]
                        assert all(not set(tr.client_id).intersection(ev.client_id) for ev in evs)
                        columns = cols_train(tr,conf,variant)
                        if not columns: raise RuntimeError('Fold không có biến hợp lệ')
                        y_train = tr.y.to_numpy(dtype='int8')
                        if variant=='shuffle':
                            y_train = np.random.default_rng(seed+fold).permutation(y_train)
                        if conf['family']!='rule':
                            if conf['k'] is not None and len(columns)>conf['k']:
                                # Xếp hạng bên trong train; fold chấm không tham gia.
                                rank_path = RUN/f'rank_{key(columns,seed,variant,anchor,tm,fold)}.json'
                                if rank_path.exists():
                                    ranked = json.loads(rank_path.read_text())
                                else:
                                    ranking = learner(cfg(trees=12 if SMOKE else 120,depth=5),seed)
                                    started = time.monotonic()
                                    ranking.fit(tr[columns].replace([np.inf,-np.inf],np.nan),y_train)
                                    record_fit(anchor,fold,seed,'rank',started)
                                    order = sorted(zip(columns,ranking.feature_importances_),key=lambda item:(-item[1],item[0]))
                                    ranked = [column for column,_ in order]
                                    rank_path.write_text(json.dumps(ranked))
                                    del ranking
                                keep = [c for c in columns if conf['prior'] and c.startswith('raw_prior_')
                                        and c.endswith(('_ever','_3m','_recency'))]
                                columns = list(dict.fromkeys(keep+ranked))[:conf['k']]
                            model = learner(conf,seed)
                            started = time.monotonic()
                            model.fit(tr[columns].replace([np.inf,-np.inf],np.nan),y_train)
                            record_fit(anchor,fold,seed,conf['family'],started)
                        rows = []
                        for ev in evs:
                            if conf['family']=='rule':
                                scores = rule(ev)
                            else:
                                x_eval = ev.reindex(columns=columns).replace([np.inf,-np.inf],np.nan)
                                if permuted_group:
                                    block = [c for c in columns if feature_group(c)==permuted_group]
                                    order = np.random.default_rng(seed+fold).permutation(len(ev))
                                    x_eval[block] = x_eval[block].to_numpy()[order]
                                scores = model.predict_proba(x_eval)[:,1]
                            rows.append(ev[['client_id','month','fold','y']].assign(score=scores))
                        parts.append(pd.concat(rows,ignore_index=True))
                        save(parts[-1],dest)
                        feat_path.write_text(json.dumps(columns),encoding='utf8')
                        n_feats.append(len(columns))
                        if conf['family']!='rule': del model
                        del tr,evs
                        gc.collect()
                bar.update()
            del train_all,eval_all
    finally:
        bar.close()
    out = pd.concat(parts,ignore_index=True)
    out.attrs['n_features'] = int(round(np.median(n_feats))) if n_feats else 0
    return out

def evaluate(name,conf,eval_months,ys,seed=42,variant='main',locked_test=False):
    say(name)
    start = time.monotonic()
    p = predict_config(conf,eval_months,ys,seed,'main' if variant=='shuffle_eval' else variant,locked_test)
    if variant=='shuffle_eval':
        p = p.copy()
        for (month,fold), part in p.groupby(['month','fold']):
            p.loc[part.index,'y'] = np.random.default_rng(seed+fold).permutation(part.y.to_numpy())
    metric_started = time.monotonic()
    rows = pd.DataFrame([dict(month=m,**metrics(part)) for m,part in p.groupby('month')])
    mean = rows.drop(columns='month').mean().to_dict()
    records.append(dict(name=name,seed=seed,variant=variant,n_features=p.attrs.get('n_features',0),
                        config=json.dumps(conf),months=','.join(eval_months),
                        metric_seconds=round(time.monotonic()-metric_started,2),
                        elapsed_seconds=round(time.monotonic()-start,2),**mean))
    pd.DataFrame(records).to_csv(RUN/'experiment_log.csv',index=False)
    display(rows.assign(run=name,n_features=p.attrs.get('n_features',0)).round(5))
    return mean,p

def passes(candidate,reference):
    return candidate['PR_AUC']>reference['PR_AUC'] and candidate['lift_1']>=.95*reference['lift_1']

RUN = FEATURE_RUN/key(BASE, [version(pkg) for pkg in ('scikit-learn','catboost','lightgbm','xgboost')],
                      code_fingerprint(feature_group,learner,cols_train,rule,predict_config,attach,metrics))
RUN.mkdir(parents=True,exist_ok=True)



## 05 · Năm baseline và kiểm chứng trên Val

Chạy 5 baseline, sau đó in kết quả kiểm chứng cutoff, DIALOG, độ trễ nhãn, tenure, overlap, nguồn GEO và xáo nhãn. LogReg và LightGBM dùng CPU; CatBoost và XGBoost dùng GPU.

In [ ]:
for family in ('rule','logreg','lgb','xgb','cat'):
    res_val, _ = evaluate(f'baseline_{family}',cfg(family=family),active_val,dev_labels)
    if family=='cat':
        base_val = res_val
screen_month = [active_val[-1]]
base_screen,_ = evaluate('cat_tho_rong_screen',BASE,screen_month,dev_labels)
has_val_prior = any(months.index(t) - int(not LABELS_READY_NEXT_MONTH) > 0 for t in fit_months[screen_month[0]])
checks = ['utc','lag7','lag14','no_dialog','prior_delay','no_prior','step3','no_geo','cap_tenure','shuffle','shuffle_eval']
if not has_val_prior:
    say('Bỏ qua audit_prior_delay và audit_no_prior trên Val: các snapshot train Val đều chưa có lịch sử prior_* (lag=1).')
    checks = [c for c in checks if c not in {'prior_delay','no_prior'}]
if SMOKE:
    checks = [c for c in ['no_prior','shuffle','shuffle_eval'] if c in checks]
for variant in checks:
    if variant=='step3' and len(fit_months[screen_month[0]])<2:
        say('Bỏ qua audit step3: cần ít nhất hai snapshot train để có lịch đối chứng.')
        continue
    evaluate(f'audit_{variant}',BASE,screen_month,dev_labels,variant=variant)

# So phân phối GEO giữa các snapshot train và kỳ chấm cuối trên cùng cohort.
geo_cols = ['raw_geo_30_count','raw_geo_30_geohash_5_unique','raw_geo_90_recency']
geo_drift = pd.DataFrame([
    {'month':m, 'feature':col, 'missing':x[col].isna().mean(),
     'zero':x[col].eq(0).mean(), 'median':x[col].median(), 'p90':x[col].quantile(.9)}
    for m in sorted(set(fit_months[screen_month[0]]+screen_month))
    for x in [panel(m)[geo_cols]]
    for col in geo_cols
])
geo_drift.to_csv(RUN/'geo_drift.csv',index=False)
display(geo_drift)



## 06 · Quét nhóm biến và K, áp dụng cổng dừng

Quét từng nhóm tín hiệu và các mức K bằng CatBoost seed 42. Dừng nhánh biến mới nếu không vượt đối chứng theo PR‑AUC và Lift@1%.

In [ ]:
groups = ['purchase','trx','geo','embedding','interactions']
if not has_val_prior:
    say('Bỏ qua nhóm purchase trên Val: chưa có lịch sử prior_* quan sát được trong train Val (LABELS_READY_NEXT_MONTH=False).')
    groups = [g for g in groups if g != 'purchase']
res = []
for group in groups:
    candidate = cfg(groups=[group])
    signal = evaluate_permutation(group,screen_month,dev_labels)
    sc,_ = evaluate(f'group_{group}',candidate,screen_month,dev_labels)
    accepted = passes(sc,base_screen)
    decision('group',group,accepted,f'Permutation delta PR-AUC={signal:.6f}; compared with wide baseline')
    if accepted:
        res.append((sc['PR_AUC'],group))
best_groups = [group for _,group in sorted(res,reverse=True)[:3]]
shortlist = []
confirmed = {}
if best_groups:
    scored = []
    anchor = screen_month[0]
    pool = pd.concat([attach(panel(m),dev_labels[m]) for m in fit_months[anchor]],ignore_index=True)
    limits = [len(cols_train(pool.loc[pool.fold.ne(fold)],cfg(groups=best_groups),'main')) for fold in range(5)]
    del pool
    seen_sizes = set()
    for k in ([20,40,None] if SMOKE else [100,150,250,None]):
        sizes = tuple(min(k,n) if k is not None else n for n in limits)
        if sizes in seen_sizes:
            decision('k',k,False,f'Duplicate effective fold sizes {sizes}')
            continue
        seen_sizes.add(sizes)
        candidate = cfg(groups=best_groups,k=k)
        sc,_ = evaluate(f'k_{k}',candidate,screen_month,dev_labels)
        accepted = passes(sc,base_screen)
        decision('k',k,accepted,'Compared with wide baseline on screening month')
        if accepted:
            scored.append((sc['PR_AUC'],candidate))
    if scored and has_val_prior:
        best_k = max(scored,key=lambda v:v[0])[1]['k']
        if best_k is not None:
            candidate = cfg(groups=best_groups,k=best_k,prior=True)
            sc,_ = evaluate(f'k_{best_k}_prior',candidate,screen_month,dev_labels)
            if passes(sc,base_screen): scored.append((sc['PR_AUC'],candidate))
    for _,candidate in sorted(scored,key=lambda v:v[0],reverse=True)[:3]:
        score, val_pred = evaluate(f'confirm_{candidate["k"]}_{candidate["prior"]}',candidate,active_val,dev_labels)
        accepted = passes(score,base_val)
        decision('confirm_k',candidate,accepted,'Compared with wide baseline across active Val')
        if accepted:
            shortlist.append((score,candidate))
            confirmed[key(candidate)] = val_pred
else:
    say('Cổng dừng: không có nhóm đặc trưng vượt CatBoost thô rộng.')
def val_delta_se(candidate, reference):
    both = candidate.merge(reference,on=['client_id','month','fold','y'],suffixes=('_a','_b'),validate='one_to_one')
    if len(both) != len(candidate) or len(both) != len(reference):
        raise ValueError('Hai cấu hình phải dự báo cùng khách/tháng')
    clients, ix = np.unique(both.client_id,return_inverse=True)
    groups = [np.flatnonzero(both.month.to_numpy()==m) for m in active_val]
    rng = np.random.default_rng(42)
    delta = []
    y,a,b = (both[col].to_numpy() for col in ('y','score_a','score_b'))
    for _ in tqdm(range(12 if SMOKE else 200),desc='Val ghép cặp 1-SE',unit='mẫu'):
        weights = rng.multinomial(len(clients), np.full(len(clients),1/len(clients)))[ix]
        if any(len(np.unique(y[g][weights[g]>0])) < 2 for g in groups):
            continue
        delta.append(np.mean([average_precision_score(y[g],a[g],sample_weight=weights[g])
                             - average_precision_score(y[g],b[g],sample_weight=weights[g]) for g in groups]))
    return float(np.std(delta,ddof=1)) if len(delta) > 1 else 0.0

if shortlist:
    best_score,best_conf = max(shortlist,key=lambda row:row[0]['PR_AUC'])
    candidates = []
    for score,conf in shortlist:
        se = val_delta_se(confirmed[key(conf)],confirmed[key(best_conf)]) if conf!=best_conf else 0
        if best_score['PR_AUC']-score['PR_AUC'] <= se and score['lift_1'] >= .95*best_score['lift_1']:
            candidates.append((score,conf))
    preferred = min(candidates,key=lambda row:(row[1]['k'] or float('inf'),-row[0]['PR_AUC']))
    shortlist = [preferred]+[row for row in shortlist if row!=preferred]
    say(f'K ưu tiên theo 1-SE: {preferred[1]["k"]}')
say(f'{len(shortlist)} tập biến mới vượt cổng trên Val')


## 07 · Tuning có ngân sách và khóa cấu hình

Chạy tối đa 12 cấu hình sàng lọc và tối đa 3 cấu hình xác nhận. Khóa số cây theo Val, không dùng fold dự báo cho early stopping.

In [ ]:
# Chốt số cây cố định qua điểm Val. Không truyền eval_set vào bất kỳ lượt fit nào.
chosen = BASE
if shortlist:
    ranked = shortlist
    candidate_sets = [item[1] for item in ranked[:2]]
    screened = []
    variants = [(fam,trees,depth) for fam in ('cat','lgb','xgb')
                for trees,depth in [(12 if SMOKE else 200,5),(12 if SMOKE else 800,6)]]
    for features in candidate_sets:
        for family,trees,depth in variants:
            conf = {**features,'family':family,'trees':trees,'depth':depth}
            if any(conf == previous for _,previous in screened):
                continue
            sc,_ = evaluate(f'tune_{len(screened)+1}',conf,screen_month,dev_labels)
            screened.append((sc,conf))
    verified = ranked[:1]
    for _,conf in sorted(screened,key=lambda v:(v[0]['PR_AUC'],v[0]['lift_1']),reverse=True)[:3]:
        sc,_ = evaluate(f'verify_{conf["family"]}_{conf["trees"]}_{conf["depth"]}',conf,active_val,dev_labels)
        accepted = passes(sc,base_val)
        decision('tuning_confirm',conf,accepted,'Compared with wide baseline across active Val')
        if accepted:
            verified.append((sc,conf))
    candidate = max(verified,key=lambda v:(v[0]['PR_AUC'],v[0]['lift_1']))
    chosen = candidate[1] if passes(candidate[0],ranked[0][0]) else ranked[0][1]
for seed in ([] if SMOKE else [142,242]):
    evaluate('val_cat_tho_rong_seed_check',BASE,active_val,dev_labels,seed)
    if chosen!=BASE:
        evaluate('val_final_seed_check',chosen,active_val,dev_labels,seed)
lock = {'run':RUN.name,'chosen':chosen,'reference':BASE,'fit_months':fit_months[test_months[0]],
        'score_months':test_months,'seeds':[42] if SMOKE else [42,142,242]}
lock['id'] = key(lock)
lock_path = RUN/'locked_config.json'
if lock_path.exists() and json.loads(lock_path.read_text()) != lock:
    raise RuntimeError('Đã có cấu hình khác trong thư mục chạy này. Dùng phiên bản RUN mới.')
lock_path.write_text(json.dumps(lock,indent=2),encoding='utf8')
display(lock)


## 08 · Mở test sau khi khóa, fit tháng 10 và chấm hai tháng

Kiểm tra cấu hình khóa rồi mới đổi `OPEN_TEST = True`. Lượt fit tháng 10 chấm cả tháng 10 và 11. Cell in bảng metric và khoảng tin cậy ghép cặp.

In [ ]:
# Chạy cell này riêng sau khi xem Cell 07. Đổi thành True để bắt đầu.
OPEN_TEST = RUN_LOCKED_TEST or SMOKE
lock_path = RUN/'locked_config.json'
report = None
if OPEN_TEST:
    if not lock_path.exists():
        raise RuntimeError('Cần hoàn tất và khóa cấu hình ở Cell 07 trước.')
    frozen = json.loads(lock_path.read_text())
    signature = frozen.pop('id')
    assert frozen['run']==RUN.name and key(frozen)==signature
    assert frozen['fit_months']==fit_months[test_months[0]] and frozen['score_months']==test_months
    receipt = RUN/'test_opened.json'
    if receipt.exists() and json.loads(receipt.read_text())!=signature:
        raise RuntimeError('Thư mục kết quả đã mở test với cấu hình khác.')
    receipt.write_text(json.dumps(signature),encoding='utf8')
    yt = labels(frozen['fit_months']+test_months)
    for m in test_months:
        panel(m)
    all_test = {}
    for seed in frozen['seeds']:
        _,ref = evaluate('test_cat_tho_rong',frozen['reference'],test_months,yt,seed,locked_test=True)
        if frozen['chosen']==frozen['reference']:
            out=ref.copy()
        else:
            _,out = evaluate('test_final',frozen['chosen'],test_months,yt,seed,locked_test=True)
        for label, result in [('CatBoost thô rộng',ref),('Mô hình chốt',out)]:
            all_test.setdefault(label,[]).append(result)
    report = pd.DataFrame([dict(model=name,seed=seed,month=m,**metrics(part))
                           for name,scores in all_test.items()
                           for seed,frame in zip(frozen['seeds'],scores)
                           for m,part in frame.groupby('month')])
    report.to_csv(RUN/'test_by_seed.csv',index=False)
    # Bootstrap ghép cặp theo khách, giữ đồng bộ mẫu giữa đối chứng và mô hình chốt.
    stats = []
    for m in test_months:
        def scores_by_customer(seeds):
            scores = [frame.loc[frame.month.eq(m)].set_index('client_id') for frame in seeds]
            values = pd.concat([x.score for x in scores],axis=1)
            assert values.notna().all().all()
            return scores[0][['y']].join(values.mean(axis=1).rename('score')).reset_index()
        left = scores_by_customer(all_test['CatBoost thô rộng'])
        right = scores_by_customer(all_test['Mô hình chốt']).set_index('client_id').loc[left.client_id].reset_index()
        assert np.array_equal(left.y,right.y)
        y_arr = left.y.to_numpy()
        sa_arr = np.clip(left.score.to_numpy(),1e-7,1-1e-7)
        sb_arr = np.clip(right.score.to_numpy(),1e-7,1-1e-7)
        ties_arr = pd.util.hash_pandas_object(left.client_id,index=False).to_numpy()
        top_n = int(np.ceil(.01*len(y_arr)))
        def fast_rank_metrics(yy, ss, tt):
            top = np.lexsort((tt,-ss))[:top_n]
            return dict(PR_AUC=average_precision_score(yy,ss),
                        lift_1=yy[top].mean()/yy.mean(),
                        ROC_AUC=roc_auc_score(yy,ss))
        point_a, point_b = fast_rank_metrics(y_arr,sa_arr,ties_arr), fast_rank_metrics(y_arr,sb_arr,ties_arr)
        rng = np.random.default_rng(42)
        delta = []
        for _ in tqdm(range(12 if SMOKE else 1000),desc=f'Bootstrap {m}',unit='mẫu'):
            ix = rng.integers(len(y_arr),size=len(y_arr))
            yy = y_arr[ix]
            if yy.min()==yy.max():
                continue
            tt = ties_arr[ix]
            a,b = fast_rank_metrics(yy,sa_arr[ix],tt), fast_rank_metrics(yy,sb_arr[ix],tt)
            delta.append({k:b[k]-a[k] for k in ('PR_AUC','lift_1','ROC_AUC')})
        for metric, bounds in pd.DataFrame(delta).quantile([.025,.975]).items():
            stats.append(dict(month=m,metric=metric,low=bounds.iloc[0],high=bounds.iloc[1],
                              delta=point_b[metric]-point_a[metric]))
    comparison = pd.DataFrame(stats)
    comparison.to_csv(RUN/'paired_bootstrap.csv',index=False)
    display(comparison.round(5))
    display(report.groupby(['model','month']).mean(numeric_only=True).drop(columns='seed').round(5))
else:
    say('Test vẫn khóa. Đổi OPEN_TEST = True trong Cell 08 rồi chạy lại riêng cell này.')


## 09 · Báo cáo và cách lưu notebook kèm đầu ra

In nhật ký thời gian, kết quả và biểu đồ tỷ lệ mua. Tải file `.ipynb` từ Colab sau khi các bảng hiện ra để giữ kết quả trong notebook.

In [ ]:
import matplotlib.pyplot as plt
log_path = RUN/'experiment_log.csv'
log = pd.DataFrame(records) if records else (pd.read_csv(log_path) if log_path.exists() else pd.DataFrame())
if len(log):
    cols_show = [c for c in ['name','months','seed','n_features','PR_AUC','lift_1','elapsed_seconds'] if c in log]
    display(log[cols_show].round(5))
    say(f'{len(fit_events)} lượt fit thực tế trong phiên này')
    if (RUN/'selection_decisions.csv').exists():
        display(pd.read_csv(RUN/'selection_decisions.csv'))
    say(f'{len(log)} lần gọi đánh giá · {log.config.nunique()} cấu hình mô hình duy nhất · {len(list(RUN.glob("pred_*.parquet")))} checkpoint dự đoán fold')
report = globals().get('report')
if report is None and (RUN/'test_by_seed.csv').exists():
    report = pd.read_csv(RUN/'test_by_seed.csv')
if report is not None:
    trend = target_rows(months).assign(y_any=lambda x:x[products].max(axis=1)).groupby('mon').y_any.mean()
    fig,ax = plt.subplots(figsize=(10,3))
    ax.plot(trend.index,trend.values,marker='o')
    ax.tick_params(axis='x',rotation=45)
    ax.set(ylabel='Tỷ lệ mua',title='Tỷ lệ khách mua theo tháng')
    from matplotlib.ticker import PercentFormatter
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    plt.show()
    display(report.round(5))
say(f'CSV, cấu hình và checkpoint: {RUN}')
print('Sau khi chạy xong: File → Download → Download .ipynb. Giữ output của cell để xem bảng và biểu đồ khi mở lại.')
